# 03 · Blind judge file and final score (dataset v4)
No GPU needed.
1. Build `judge_items.csv` (systems mixed, shuffled, system names hidden, 10% hidden duplicates). Answers that the abstention pattern catches ("Bilmiyorum", "belirtilmemiş", "bilgi yok", ...) are scored automatically. Every other answer goes to the judge, including answers on "Bilmiyorum." items, and the judge flags (`abstains = 1`) any answer that says the story does not give the information in other words.
2. Send **only** `judge_items.csv` to the judge. Keep `judge_key.csv` and `all_predictions.csv` private.
3. Put the returned `judge_scores.csv` in the same folder and run Step 2. It writes `report.md`, `report.json`, `per_item.csv` and `results.png`.

**Scoring.** Abstained = caught by the pattern **or** flagged by the judge. Answerable item: judge score 2 / 1 / 0; abstaining on it counts as 0. "Bilmiyorum." item: 2 if the model abstained, in any wording, else 0 (any content answer is a made-up fact). The report shows how many abstentions each route found. **Overall score** = mean of score / 2 over all 668 items. Also reported: judged accuracy on answerable items (overall, past, future, and the 350 v3-test pairs), abstention precision / recall / F1, recall per source, false abstention rate, and a cue-only reference F1 (abstain on hangi / kaç / ne kadar questions).

## Setup

In [ ]:
# ---- Settings (edit once) ----
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
LOCAL = "/content/data"            # local copy of the dataset (Drive I/O is slow)

In [ ]:
# ---- Mount Drive ----
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
!pip install -q sacrebleu==2.5.1
import os, json
import numpy as np, pandas as pd

In [ ]:
# ---- Copy the frozen dataset (v4) to local disk and check it is unchanged ----
import hashlib, shutil
DATA_VERSION = "v4"
DATA_FILES = {   # sha256 from data/v4/MANIFEST.json (frozen 2026-11-26)
    "splits/train.csv": "a6832dc2104f430c684e961c08be6e51f075ead1559d2559905f38b871de1053",
    "splits/test.csv": "6f85e02b5d8e5dc9d90d8eb30137342fd02a3943302fbe286f574474681b6a7c",
    "structured/structured.csv": "e133b681b2e77c467020b58123e6dc60d7fd576ce25cc500c7a290285a43bbc9",
}
for rel, sha in DATA_FILES.items():
    dst = f"{LOCAL}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f"{DRIVE_ROOT}/data/{DATA_VERSION}/{rel}", dst)
    got = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    assert got == sha, f"STOP: {rel} differs from the frozen {DATA_VERSION} file"
print("dataset OK:", DATA_VERSION, ", ".join(DATA_FILES))


## Code

In [ ]:
# ---- Chart style: categorical slots 1/2 for the two systems ----
C_BASE, C_FT = "#2a78d6", "#eb6834"
C_TEXT, C_MUTED, C_GRID = "#0b0b0b", "#52514e", "#e4e3df"


def style_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(C_MUTED)
    ax.tick_params(colors=C_MUTED, labelsize=9)
    ax.grid(axis="y", color=C_GRID, linewidth=0.8)
    ax.set_axisbelow(True)

In [ ]:
# ---- Build one blind judge file from several systems' predictions ----
import re
import numpy as np
import pandas as pd

# Abstention = the answer says the text does not give the information. Same pattern in notebooks 01, 02 and 03.
# The pattern is a first pass only: in notebook 03 the judge also flags abstentions written in other words.
# Word boundaries keep story answers such as "bilmiyordu" (he did not know) from counting as abstention.
ABSTAIN = "Bilmiyorum."
_ABST = re.compile(
    r"\b(bilmiyorum|bilemiyorum|bilemem|bilinmiyor|bilinemez|belirtilmemiş|belirtilmiyor|söylenmemiş|söylenmiyor"
    r"|bahsedilmiyor|bahsedilmemiş|söz edilmiyor|söz edilmemiş|cevap veremem|çıkarılamaz|çıkarılamıyor|tahmin edilemez)\b"
    r"|bilgi (yok|verilmemiş|verilmiyor|bulunmuyor|yer almıyor)|yeterli (bilgi|ipucu)"
    r"|(metinde|metinden|hikâyede|hikayede|bölümde|bölümden) (yer almıyor|geçmiyor|anlaşılmıyor|yok)\b|\bbelli değil\b")


def tr_lower(s):
    return str(s).replace("I", "ı").replace("İ", "i").lower()


def is_abstention(text):
    return bool(_ABST.search(tr_lower(text)))


def make_judge_file(preds, out_dir, split="test", dup_frac=0.10, seed=42, data_dir=LOCAL):
    """preds: {"baseline": path, "finetuned": path, ...}.
    Answers that the abstention pattern catches are scored automatically in Step 2. Every other answer goes to the
    judge, also on Bilmiyorum items (gold "Bilmiyorum."), and the judge flags any abstention written in other words.
    Writes judge_items.csv (send to the judge) and judge_key.csv + all_predictions.csv (keep private).
    A share of judged items (dup_frac) is repeated under new ids to measure judge self-consistency."""
    ref = pd.read_csv(f"{data_dir}/splits/{split}.csv", keep_default_na=False)
    st = pd.read_csv(f"{data_dir}/structured/structured.csv")[["text_id", "part1", "part3"]]
    ref = ref.merge(st, on="text_id", validate="many_to_one")
    assert ref.qid.is_unique
    rows = []
    for name, path in preds.items():
        p = pd.read_csv(path, keep_default_na=False)[["qid", "prediction"]]
        m = ref.merge(p, on="qid", how="left", validate="one_to_one")
        miss = m.prediction.isna().sum()
        assert miss == 0, f"{name}: {miss} {split} items have no prediction (old predictions without qid? re-run predict)"
        m["system"] = name
        m["abstained_pattern"] = m.prediction.map(is_abstention).astype(int)
        rows.append(m)
    allp = pd.concat(rows, ignore_index=True)
    for name, g in allp.groupby("system", sort=False):
        print(f"{name}: pattern found abstentions on {g[g.answerable == 0].abstained_pattern.mean():.1%} of Bilmiyorum items, "
              f"{g[g.answerable == 1].abstained_pattern.mean():.1%} of answerable items (judge adds paraphrases)")

    jud = allp[allp.abstained_pattern == 0].copy().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    jud["uid"] = range(len(jud))
    jud["dup_uid"] = -1
    n_dup = int(round(dup_frac * len(jud)))
    if n_dup:
        dup = jud.iloc[rng.choice(len(jud), n_dup, replace=False)].copy()
        dup["dup_uid"] = dup["uid"]
        dup["uid"] = range(len(jud), len(jud) + n_dup)
        jud = pd.concat([jud, dup], ignore_index=True)
    jud = jud.iloc[rng.permutation(len(jud))].reset_index(drop=True)
    jud["item_id"] = [f"J{i + 1:04d}" for i in range(len(jud))]
    uid2item = dict(zip(jud.uid, jud.item_id))
    jud["dup_of"] = jud.dup_uid.map(lambda u: uid2item[u] if u >= 0 else "")

    os.makedirs(out_dir, exist_ok=True)
    jud[["item_id", "direction", "part1", "part2", "part3", "question", "answer", "prediction"]] \
        .rename(columns={"answer": "gold_answer", "prediction": "candidate_answer"}) \
        .to_csv(f"{out_dir}/judge_items.csv", index=False)
    jud[["item_id", "system", "qid", "dup_of"]].to_csv(f"{out_dir}/judge_key.csv", index=False)
    allp[["system", "qid", "text_id", "direction", "answerable", "source", "in_v3_test", "question", "answer",
          "prediction", "abstained_pattern"]].to_csv(f"{out_dir}/all_predictions.csv", index=False)
    print(f"{len(jud)} judge items ({len(jud) - n_dup} unique + {n_dup} hidden duplicates) from {len(preds)} systems; "
          f"{len(allp) - (len(jud) - n_dup)} items scored automatically -> {out_dir}")
    print("send judge_items.csv only; keep judge_key.csv and all_predictions.csv private")


In [ ]:
# ---- Turn judge scores into the final numbers ----
# judge_scores.csv columns: item_id, score (2 = correct, 1 = partial, 0 = wrong),
#                           abstains (1 = the answer says the story does not give the information, in any words), reason (optional)
import json


def tokens(s):
    return re.findall(r"\w+", tr_lower(s))


def rouge_l_f1(pred, ref):
    p, r = tokens(pred), tokens(ref)
    if not p or not r:
        return 0.0
    dp = [[0] * (len(r) + 1) for _ in range(len(p) + 1)]
    for i in range(len(p)):
        for j in range(len(r)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if p[i] == r[j] else max(dp[i][j + 1], dp[i + 1][j])
    lcs = dp[-1][-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)


def chrf(preds, refs):
    from sacrebleu.metrics import CHRF
    return CHRF().corpus_score(list(preds), [list(refs)]).score


def question_type(q):
    q = tr_lower(q)
    for name, pat in [("how many", r"\bkaç|ne kadar"), ("which", r"\bhangi"), ("when", r"ne zaman"),
                      ("where", r"\bnere"), ("who", r"\bkim"), ("why", r"\b(neden|niçin|niye)\b"),
                      ("how", r"\bnasıl"), ("yes/no", r"\bm[ıiuü](yd[ıiuü]|s[ıiuü]n)?\b"), ("what", r"\bne")]:
        if re.search(pat, q):
            return name
    return "other"


CUE = re.compile(r"\bhangi|\bkaç|ne kadar")   # question-type shortcut (see dataset README)


def abst_prf(abst, unans):
    tp = int((abst & unans).sum())
    p = tp / abst.sum() if abst.sum() else 0.0
    r = tp / unans.sum() if unans.sum() else 0.0
    return float(p), float(r), float(2 * p * r / (p + r)) if p + r else 0.0


def paired_bootstrap(stat, n=10000, seed=42, size=None):
    """95% CI of stat(idx) over resampled item indices; stat returns compare - baseline for an index matrix."""
    rng = np.random.default_rng(seed)
    d = stat(rng.integers(0, size, size=(n, size)))
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def mcnemar_exact(a_ok, b_ok):
    from scipy.stats import binomtest
    b01 = int(((~a_ok) & b_ok).sum())   # baseline wrong, compare right
    b10 = int((a_ok & ~b_ok).sum())     # baseline right, compare wrong
    p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    return b01, b10, float(p)


def acc_block(g):
    """g: answerable items of one system. Abstentions count as wrong."""
    s = g.score
    return {"n": int(len(g)), "judged_accuracy": float((s / 2).mean()), "strict_accuracy": float((s == 2).mean()),
            "lenient_accuracy": float((s >= 1).mean()), "correct": int((s == 2).sum()), "partial": int((s == 1).sum()),
            "wrong": int(((s == 0) & (g.abstained == 0)).sum()), "abstained": int(g.abstained.sum())}


def summarize(g):
    a, u = g[g.answerable == 1], g[g.answerable == 0]
    ans = a[a.abstained == 0]
    p, r, f = abst_prf(g.abstained.values == 1, g.answerable.values == 0)
    out = {
        "n_items": int(len(g)),
        "overall_score": float(g.item_score.mean()),        # all items: answerable = judge score / 2, Bilmiyorum = 1 if abstained
        "answerable": {**acc_block(a),
                       "false_abstention_rate": float(a.abstained.mean()),
                       "judged_accuracy_when_answered": float((ans.score / 2).mean()) if len(ans) else None,
                       "rougeL_f1": float(np.mean([rouge_l_f1(x, y) for x, y in zip(a.prediction, a.answer)])),
                       "chrF": chrf(a.prediction.astype(str), a.answer.astype(str))},
        "by_direction": {d: acc_block(gg) for d, gg in a.groupby("direction")},
        "v3_test_subset": acc_block(a[a.in_v3_test == 1]),
        "bilmiyorum_items": {"n": int(len(u)), "abstained": int(u.abstained.sum()), "recall": float(u.abstained.mean()),
                             **{f"recall_{s}": float(gg.abstained.mean()) for s, gg in u.groupby("source")}},
        "abstention": {"predicted": int(g.abstained.sum()), "precision": p, "recall": r, "f1": f},
    }
    return out


def score(judge_dir, scores_path, comparisons=(("baseline", "finetuned"),)):
    """Writes report.md, report.json, per_item.csv (private) and results.png in judge_dir."""
    items = pd.read_csv(f"{judge_dir}/judge_items.csv", keep_default_na=False)
    key = pd.read_csv(f"{judge_dir}/judge_key.csv", keep_default_na=False)
    allp = pd.read_csv(f"{judge_dir}/all_predictions.csv", keep_default_na=False)
    sc = pd.read_csv(scores_path, keep_default_na=False)
    assert set(sc.item_id) == set(items.item_id), "scores must cover every judge item exactly once"
    assert sc.item_id.is_unique
    sc["score"] = sc.score.astype(int)
    assert sc.score.isin([0, 1, 2]).all(), "score must be 0, 1 or 2"
    assert "abstains" in sc, "judge_scores.csv needs an abstains column (0/1) for every item"
    sc["abstains"] = sc.abstains.astype(int)
    assert sc.abstains.isin([0, 1]).all(), "abstains must be 0 or 1"
    jd = key.merge(sc[["item_id", "score", "abstains"]], on="item_id")

    # judge self-consistency on hidden duplicates (not independent: the judge may remember the first rating)
    dups = jd[jd.dup_of != ""]
    orig = jd.set_index("item_id").score
    cons = {"n_duplicates": int(len(dups))}
    if len(dups):
        same = dups.score.values == orig.loc[dups.dup_of].values
        cons.update(exact_agreement=float(same.mean()),
                    keep_drop_agreement=float(((dups.score.values >= 1) == (orig.loc[dups.dup_of].values >= 1)).mean()))
    jd = jd[jd.dup_of == ""]

    df = allp.merge(jd[["system", "qid", "score", "abstains"]], on=["system", "qid"], how="left", validate="one_to_one")
    sent = df.abstained_pattern == 0
    assert df.loc[sent, "score"].notna().all(), "a judged item has no score"
    assert df.loc[~sent, "score"].isna().all(), "a pattern-caught item was sent to the judge"
    # abstention = caught by the pattern OR flagged by the judge (any wording)
    df["abstained_judge"] = df.abstains.fillna(0).astype(int)
    df["abstained"] = ((df.abstained_pattern == 1) | (df.abstained_judge == 1)).astype(int)
    df["abstain_found_by"] = np.select([df.abstained_pattern == 1, df.abstained_judge == 1], ["pattern", "judge"], "")
    u = df.answerable == 0
    flag_disagree = int((sent & u & ((df.score == 2) != (df.abstained_judge == 1))).sum())   # judge's two fields should agree here
    df.loc[(df.answerable == 1) & (df.abstained == 1), "score"] = 0      # abstained on an answerable item
    df.loc[u, "score"] = 2 * df.loc[u, "abstained"]                      # Bilmiyorum item: abstained = correct
    df["score"] = df.score.astype(int)
    df = df.drop(columns="abstains")
    df["item_score"] = df.score / 2
    df["qtype"] = df.question.map(question_type)
    df.to_csv(f"{judge_dir}/per_item.csv", index=False)

    systems = list(dict.fromkeys(df.system))
    report = {"judge_consistency": cons, "systems": {s: summarize(df[df.system == s]) for s in systems},
              "abstentions_found": {s: df[df.system == s].abstain_found_by.value_counts().drop("", errors="ignore").to_dict()
                                    for s in systems},
              "judge_score_vs_abstains_disagreements_on_bilmiyorum_items": flag_disagree}
    ref = df[df.system == systems[0]]
    cue = ref.question.map(lambda q: bool(CUE.search(tr_lower(q)))).values
    report["cue_only_reference"] = dict(zip(["precision", "recall", "f1"], abst_prf(cue, ref.answerable.values == 0)))
    report["abstention_by_question_type"] = {
        s: {t: {"n_bilmiyorum": int((g.answerable == 0).sum()), "recall": float(g[g.answerable == 0].abstained.mean()) if (g.answerable == 0).any() else None,
                "n_answerable": int((g.answerable == 1).sum()), "false_abstention": float(g[g.answerable == 1].abstained.mean()) if (g.answerable == 1).any() else None}
            for t, g in df[df.system == s].groupby("qtype")} for s in systems}

    report["comparisons"] = []
    for base, comp in comparisons:
        if base not in systems or comp not in systems:
            continue
        b = df[df.system == base].set_index("qid").sort_index()
        c = df[df.system == comp].set_index("qid").sort_index()
        assert b.index.equals(c.index), "systems were not scored on the same items"
        bs, cs = b.item_score.values, c.item_score.values
        am = (b.answerable == 1).values
        ba, ca = bs[am], cs[am]
        unans = ~am
        bab, cab = b.abstained.values == 1, c.abstained.values == 1

        def f1_delta(idx):
            out = []
            for row in idx:
                out.append(abst_prf(cab[row], unans[row])[2] - abst_prf(bab[row], unans[row])[2])
            return np.array(out)

        ov_ci = paired_bootstrap(lambda idx: cs[idx].mean(1) - bs[idx].mean(1), size=len(bs))
        an_ci = paired_bootstrap(lambda idx: ca[idx].mean(1) - ba[idx].mean(1), size=len(ba))
        f1_ci = paired_bootstrap(f1_delta, n=2000, size=len(bs))
        b01, b10, p = mcnemar_exact(bs == 1, cs == 1)
        lab = lambda v: np.select([v == 1, v == 0.5], [2, 1], 0)
        trans = pd.crosstab(lab(bs), lab(cs)).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
        sb, sc_ = report["systems"][base], report["systems"][comp]
        report["comparisons"].append({
            "baseline": base, "compare": comp,
            "overall_score": [sb["overall_score"], sc_["overall_score"]],
            "overall_delta_points": float(100 * (cs.mean() - bs.mean())), "overall_delta_95ci": [100 * ov_ci[0], 100 * ov_ci[1]],
            "answerable_acc": [sb["answerable"]["judged_accuracy"], sc_["answerable"]["judged_accuracy"]],
            "answerable_delta_points": float(100 * (ca.mean() - ba.mean())), "answerable_delta_95ci": [100 * an_ci[0], 100 * an_ci[1]],
            "abstention_f1": [sb["abstention"]["f1"], sc_["abstention"]["f1"]],
            "abstention_f1_delta": sc_["abstention"]["f1"] - sb["abstention"]["f1"], "abstention_f1_delta_95ci": list(f1_ci),
            "mcnemar_strict_all_items": {"baseline_wrong_compare_right": b01, "baseline_right_compare_wrong": b10, "p": p},
            "meets_10_points_overall": bool(100 * (cs.mean() - bs.mean()) >= 10),
            "transitions_all_items": {"rows_baseline_cols_compare_wrong_partial_correct": trans.values.tolist()},
        })

    json.dump(report, open(f"{judge_dir}/report.json", "w"), indent=2, ensure_ascii=False)
    L = ["# Judged evaluation (dataset v4, with Bilmiyorum items)", "",
         "Overall score = mean over all items: answerable item = judge score / 2 (abstained = 0); Bilmiyorum item = 1 if the model abstained, else 0.", "",
         "| system | overall score | answerable acc | strict | past | future | v3-test subset (350) | false abstention | Bilmiyorum recall | abst. precision | abst. F1 |",
         "|---|---|---|---|---|---|---|---|---|---|---|"]
    pc = lambda v: "n/a" if v is None else f"{100 * v:.1f}"
    for s, m in report["systems"].items():
        a, d = m["answerable"], m["by_direction"]
        L.append(f"| {s} | {pc(m['overall_score'])} | {pc(a['judged_accuracy'])} | {pc(a['strict_accuracy'])} | "
                 f"{pc(d.get('past', {}).get('judged_accuracy'))} | {pc(d.get('future', {}).get('judged_accuracy'))} | "
                 f"{pc(m['v3_test_subset']['judged_accuracy'])} | {pc(a['false_abstention_rate'])} | "
                 f"{pc(m['bilmiyorum_items']['recall'])} | {pc(m['abstention']['precision'])} | {pc(m['abstention']['f1'])} |")
    cr = report["cue_only_reference"]
    L += ["", f"Cue-only reference (abstain if the question contains hangi / kaç / ne kadar): precision {pc(cr['precision'])}, "
          f"recall {pc(cr['recall'])}, F1 {pc(cr['f1'])}.", "",
          "| system | Bilmiyorum recall: filter2_drop | Bilmiyorum recall: new_not_in_story | answerable acc when answered | ROUGE-L | chrF |",
          "|---|---|---|---|---|---|"]
    for s, m in report["systems"].items():
        u, a = m["bilmiyorum_items"], m["answerable"]
        L.append(f"| {s} | {pc(u.get('recall_filter2_drop'))} | {pc(u.get('recall_new_not_in_story'))} | "
                 f"{pc(a['judged_accuracy_when_answered'])} | {pc(a['rougeL_f1'])} | {a['chrF']:.1f} |")
    for c in report["comparisons"]:
        L += ["", f"**{c['compare']} vs {c['baseline']}:** overall score {c['overall_delta_points']:+.1f} points "
              f"(95% CI {c['overall_delta_95ci'][0]:+.1f} to {c['overall_delta_95ci'][1]:+.1f}); "
              f"answerable accuracy {c['answerable_delta_points']:+.1f} points "
              f"(95% CI {c['answerable_delta_95ci'][0]:+.1f} to {c['answerable_delta_95ci'][1]:+.1f}); "
              f"abstention F1 {100 * c['abstention_f1_delta']:+.1f} points "
              f"(95% CI {100 * c['abstention_f1_delta_95ci'][0]:+.1f} to {100 * c['abstention_f1_delta_95ci'][1]:+.1f}). "
              f"McNemar (strict, all items) p = {c['mcnemar_strict_all_items']['p']:.4f}."]
    L += ["", "Abstentions found by the pattern / added by the judge (paraphrases): " + "; ".join(
        f"{s} {v.get('pattern', 0)} / {v.get('judge', 0)}" for s, v in report["abstentions_found"].items())
          + f". Judge score and abstains flag disagree on {flag_disagree} Bilmiyorum items (check these in per_item.csv)."]
    if cons["n_duplicates"]:
        L += ["", f"Judge self-consistency on {cons['n_duplicates']} hidden duplicates: "
              f"{100*cons['exact_agreement']:.1f}% same score, {100*cons['keep_drop_agreement']:.1f}% same pass/fail."]
    open(f"{judge_dir}/report.md", "w", encoding="utf-8").write("\n".join(L) + "\n")
    print("\n".join(L))
    plot_results(report, f"{judge_dir}/results.png")
    return report


In [ ]:
# ---- Results figure (also redraws from a saved report.json) ----
C_EXTRA = ["#8a8984", "#b5b4ae"]   # neutral grays for any system beyond the first comparison pair


def plot_results(report, path):
    """Three panels: answerable accuracy, abstention behaviour, outcome on answerable items."""
    import matplotlib.pyplot as plt
    S = report["systems"]
    first = report["comparisons"][0] if report.get("comparisons") else None
    order = [first["baseline"], first["compare"]] if first else []
    order += [s for s in S if s not in order]
    colors = dict(zip(order, [C_BASE, C_FT] + C_EXTRA if first else C_EXTRA + [C_BASE, C_FT]))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), dpi=150, gridspec_kw={"width_ratios": [1.25, 1.25, 1]})
    w = 0.8 / len(order)

    def grouped(ax, labels, getter, title, ylabel):
        x = np.arange(len(labels))
        for k, s in enumerate(order):
            vals = [100 * (getter(S[s], l) or 0) for l in labels]
            bars = ax.bar(x + (k - (len(order) - 1) / 2) * w, vals, w * 0.92, color=colors[s], label=s)
            ax.bar_label(bars, fmt="%.0f", fontsize=7, color=C_TEXT, padding=2)
        ax.set_xticks(x, [l[0] for l in labels], fontsize=8)
        ax.set_ylim(0, 115)
        ax.set_ylabel(ylabel, color=C_MUTED, fontsize=9)
        ax.set_title(title, loc="left", fontsize=11, color=C_TEXT)
        ax.legend(frameon=False, fontsize=8, ncol=len(order), loc="upper left")
        style_axes(ax)

    # (a) score and accuracy
    acc = [("overall\nscore", lambda m: m["overall_score"]),
           ("answerable", lambda m: m["answerable"]["judged_accuracy"]),
           ("past", lambda m: m["by_direction"].get("past", {}).get("judged_accuracy")),
           ("future", lambda m: m["by_direction"].get("future", {}).get("judged_accuracy")),
           ("v3-test\nsubset", lambda m: m["v3_test_subset"]["judged_accuracy"])]
    grouped(axes[0], acc, lambda m, l: l[1](m), "Score and judged accuracy", "%")

    # (b) when does the model say Bilmiyorum?
    ab = [("Bilmiyorum\nitems", lambda m: m["bilmiyorum_items"]["recall"]),
          ("filter2\ndrop", lambda m: m["bilmiyorum_items"].get("recall_filter2_drop")),
          ("not in\nstory", lambda m: m["bilmiyorum_items"].get("recall_new_not_in_story")),
          ("answerable\n(false)", lambda m: m["answerable"]["false_abstention_rate"]),
          ("abstention\nF1", lambda m: m["abstention"]["f1"])]
    grouped(axes[1], ab, lambda m, l: l[1](m), "Abstention: share answered \"Bilmiyorum\", and F1", "%")
    cue = report.get("cue_only_reference", {}).get("f1")
    if cue is not None:
        axes[1].plot([3.55, 4.45], [100 * cue] * 2, color=C_TEXT, linestyle="--", linewidth=1.5)
        axes[1].annotate(f"cue-only F1 {100 * cue:.0f}", (4, 100 * cue), xytext=(0, 4), textcoords="offset points",
                         ha="center", fontsize=7, color=C_TEXT,
                         bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none"))

    # (c) outcome on answerable items (ordered levels: one blue ramp, abstained in gray)
    ax = axes[2]
    levels = [("abstained", "#d3d2cc"), ("wrong", "#cde2fb"), ("partial", "#6da7ec"), ("correct", "#1c5cab")]
    left = np.zeros(len(order))
    for lvl, col in levels:
        v = np.array([100 * S[s]["answerable"][lvl] / S[s]["answerable"]["n"] for s in order])
        ax.barh(order, v, left=left, color=col, edgecolor="white", linewidth=2, label=lvl, height=0.55)
        for i, (l, vi) in enumerate(zip(left, v)):
            if vi >= 6:
                ax.text(l + vi / 2, i, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if lvl in ("partial", "correct") else C_TEXT)
        left += v
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("share of answerable test items (%)", color=C_MUTED, fontsize=9)
    ax.set_title("Outcome on answerable items", loc="left", fontsize=11, color=C_TEXT)
    ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    style_axes(ax)
    ax.grid(False)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.show()


## Step 1: build the judge file

In [ ]:
RUN_ID = "PASTE_V4_RUN_ID_HERE"     # folder name under runs/ (the v4 run)
JUDGE = f"{DRIVE_ROOT}/eval/judge_{RUN_ID}"
SYSTEMS = {"baseline": f"{DRIVE_ROOT}/eval/baseline_untrained_v4/predictions_test.csv",
           "finetuned": f"{DRIVE_ROOT}/runs/{RUN_ID}/predictions_test.csv"}
# optional, if notebook 02 ran the v3 adapter on the v4 test set:
if os.path.exists(f"{DRIVE_ROOT}/eval/finetuned_v3_on_v4/predictions_test.csv"):
    SYSTEMS["finetuned_v3"] = f"{DRIVE_ROOT}/eval/finetuned_v3_on_v4/predictions_test.csv"
make_judge_file(SYSTEMS, JUDGE)
print("send this file to the judge:", f"{JUDGE}/judge_items.csv")

## Step 2: after `judge_scores.csv` is in the folder

In [ ]:
RUN_ID = "PASTE_V4_RUN_ID_HERE"     # same as Step 1 (repeated so Step 2 can run on its own)
JUDGE = f"{DRIVE_ROOT}/eval/judge_{RUN_ID}"
report = score(JUDGE, f"{JUDGE}/judge_scores.csv",
               comparisons=(("baseline", "finetuned"), ("finetuned_v3", "finetuned")))   # pairs with a missing system are skipped

Already scored? Redraw the figure from the saved report without re-scoring:

In [ ]:
plot_results(json.load(open(f"{JUDGE}/report.json")), f"{JUDGE}/results.png")